In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost

In [4]:
feature = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    'tolls_amount',
]

label = "tip_amount"

In [25]:
def load_datset():
    path = "yellow_tripdata_2025-02.parquet"
    df = pd.read_parquet(path, columns=feature + [label])
    X_train, X_test, y_train, y_test = train_test_split(df[feature],df[label], test_size=0.2)
    
    dtrain = xgboost.DMatrix(X_train, label=y_train)
    dtest = xgboost.DMatrix(X_test, label=y_test)
    return dtrain, dtest
    

In [26]:
storage_folder = "/tmp/xgboost_ray_example/"

In [34]:
from pathlib import Path
model_path = Path(storage_folder) / "xgboost_model.ubj"

model_path.parent.mkdir(parents=True, exist_ok=True)

def xg_boost_train(params):
    dtrain, dtest = load_datset()
    evals = [(dtest, "eval")]
    evals_results = {}
    bst = xgboost.train(
        params,
        dtrain,
        num_boost_round=10,
        evals=evals,
        evals_result = evals_results
    )
    bst.save_model(str(model_path))
    return {"eval-rmse": evals_results["eval"]["rmse"][-1]}

In [35]:
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "max_depth": 6,
    "eta": 0.1
}

xg_boost_train(params)

[0]	eval-rmse:3.47745
[1]	eval-rmse:3.33388
[2]	eval-rmse:3.21279
[3]	eval-rmse:3.11022
[4]	eval-rmse:3.02513
[5]	eval-rmse:2.95294
[6]	eval-rmse:2.89389
[7]	eval-rmse:2.84437
[8]	eval-rmse:2.80395
[9]	eval-rmse:2.76935


{'eval-rmse': 2.769354285024687}

In [ ]:
print(model_path.resolve())

S:\tmp\xgboost_ray_example\xgboost_model.ubj


: 